# QA thresholds from the validation visits

Suggested `WARN`/`FAIL` thresholds for the `imageQualityQa` metrics, derived from the
[validation visits](validation-visits.md) and plotted against the known-good and known-bad
data.

**How to run it:** follow [`deriving-thresholds.md`](deriving-thresholds.md). It runs
`imageQualityQa` over the validation visits into a new collection, then this notebook with
`QA_VALIDATION_COLLECTION` naming that collection. Step 5 there says how to read the results,
and step 6 how to adopt them.

The notebook reads the Butler read-only, once, and caches the metrics as parquet, one file
per collection.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfs.drp.qa.metrics.calibration import DEFAULT_METRICS, METRIC_SPECS, calibrate, labelRows
from pfs.drp.qa.metrics.readers import readMetrics
from pfs.drp.qa.metrics.validationVisits import loadValidationVisits, unmatchedEntries
from pfs.drp.qa.plotting import plotThresholds

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

## Parameters

In [ ]:
BUTLER = "/work/datastore"
COLLECTION = os.environ["QA_VALIDATION_COLLECTION"]
CACHE = (
    Path(os.environ.get("DRP_QA_CACHE", "~/.cache/drp_qa")).expanduser()
    / f"iqQaMetrics-{COLLECTION.replace('/', '_')}.parquet"
)
METRICS = [*DEFAULT_METRICS, "nLines"]
print(f"collection: {COLLECTION}")
print(f"cache: {CACHE}")

## Read the metrics

In [ ]:
visitSet = loadValidationVisits()
print(
    f"{len(visitSet.knownGood)} known_good entries ({len(visitSet.goodVisits)} visits), "
    f"{len(visitSet.confirmedBad)} known_bad, {len(visitSet.unconfirmedBad)} unconfirmed"
)

if CACHE.exists():
    metrics = pd.read_parquet(CACHE)
else:
    from lsst.daf.butler import Butler

    butler = Butler(BUTLER, collections=[COLLECTION], writeable=False)
    metrics = readMetrics(butler, visitSet.visits, collections=[COLLECTION])
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    metrics.to_parquet(CACHE)
print(f"{len(metrics)} rows over {metrics['visit'].nunique()} visits from {CACHE}")
missing = sorted(set(visitSet.visits) - set(metrics["visit"]))
print(f"{len(missing)} validation visits have no metrics: {missing}")

# An entry that matches no row is a check that silently does not run: its
# visits were not processed, or a selector (often seqType) is wrong.
for entry in unmatchedEntries(metrics, visitSet):
    print(
        f"matches no rows: visits {entry.visits[0]}-{entry.visits[-1]} "
        f"seqType={entry.seqType!r} expect={entry.expect}"
    )

## Suggested thresholds

One row per metric and population. Columns to read before trusting a row:
`nGood`/`nGoodVisits` (sample size), `failBounded` (whether the sample bounds the FAIL
percentile), `goodFlaggedWarn`/`goodFlaggedFail` (what the thresholds flag of the good
data), `degenerate`, and `badOk` (whether the known-bad rows reach their expected verdict;
empty when none name the metric in that population).

In [ ]:
table = calibrate(metrics, visitSet, METRICS)
columns = [
    "metric",
    "group",
    "nGood",
    "nGoodVisits",
    "median",
    "robustRms",
    "warn",
    "fail",
    "failLow",
    "failHigh",
    "goodFlaggedWarn",
    "goodFlaggedFail",
    "reliable",
    "failBounded",
    "degenerate",
    "nBad",
    "badOk",
    "nUnconfirmed",
    "unconfirmedFlagged",
]
table[columns].round(4)

## Known-bad checks

Every population with known-bad rows, and what they reached.

In [ ]:
checked = table[table["nBad"] > 0]
for _, row in checked.iterrows():
    print(f"{row['metric']}[{row['group']}]: {row['badSummary']}")

## Plots

Per population: the CDF of the known-good values, WARN (amber) and FAIL (red) with the
95 % interval on the FAIL percentile shaded, and below it the known-bad values (crosses
expect FAIL, triangles WARN) and the unconfirmed ones (open circles). A known-bad marker
left of FAIL (right of it, for `nLines`) is a fault the threshold misses.

In [ ]:
for metric in METRICS:
    labelled = labelRows(metrics, visitSet, metric)
    plotThresholds(labelled, table, metric, absolute=METRIC_SPECS[metric].absolute)
    plt.show()

## Provenance

The sentences for the config fields' `doc`, for the suggestions you adopt.

In [ ]:
for _, row in table[table["nGood"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: warn={row['warn']:g} fail={row['fail']:g}")
    print(f"    {row['provenance']}")